# Vector stores and semantic search



In [ ]:
pip install sentence-transformers numpy pandas datasets

In [ ]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray = None

    def add_documents(self, documents: list[Document]):
        # Añadir documentos a la lista en memoria
        self.documents.extend(documents)
        
        # Extraer los textos y generar embeddings en batch
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts)
        
        # Apilar los nuevos embeddings en nuestra matriz NumPy
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        # 1. Vectorizar la consulta
        query_embedding = self.embedding_model.encode([query])[0]
        
        # 2. Calcular Similitud del Coseno vectorizada
        dot_products = np.dot(self.embeddings, query_embedding)
        norms_docs = norm(self.embeddings, axis=1)
        norm_query = norm(query_embedding)
        
        # Evitar división por cero
        norms_docs[norms_docs == 0] = 1e-10
        similarities = dot_products / (norms_docs * norm_query)
        
        # 3. Obtener los índices de los 'top_k' resultados más altos
        # argsort ordena de menor a mayor, por lo que invertimos [::-1]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        # 4. Construir y retornar los resultados
        return [SearchResult(float(similarities[idx]), self.documents[idx]) for idx in top_indices]

# ==========================================
# Ejecución y Pruebas - Parte I
# ==========================================

print("Cargando modelo...")
model = SentenceTransformer('all-MiniLM-L6-v2')
store = VectorStore(model)

print("Descargando Animal Fun Facts Dataset...")
url = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"
df_animals = pd.read_csv(url)

# Manejar valores nulos para evitar errores en metadatos
df_animals = df_animals.fillna("")

documents_to_add = []
for _, row in df_animals.iterrows():
    metadata = {
        "animal_name": str(row.get("animal_name", "")),
        "source": str(row.get("source", "")),
        "media_link": str(row.get("media_link", "")),
        "wikipedia_link": str(row.get("wikipedia_link", ""))
    }
    # Asumimos que la columna de texto se llama 'text' en el dataset original
    doc = Document(text=str(row.get("text", "")), metadata=metadata)
    documents_to_add.append(doc)

print(f"Indexando {len(documents_to_add)} documentos...")
store.add_documents(documents_to_add)

consultas_animales = [
    "What animal sleeps standing up?",
    "Birds that cannot fly but swim very well",
    "Which animal has the strongest bite force?",
    "Facts about marine mammals communicating",
    "Creatures that can change their skin color"
]

print("\n--- Resultados Parte I ---")
for q in consultas_animales:
    print(f"\nConsulta: '{q}'")
    resultados = store.search(q, top_k=2)
    for i, res in enumerate(resultados):
        animal = res.document.metadata.get('animal_name', 'Unknown')
        print(f"  [{i+1}] Score: {res.score:.4f} | Animal: {animal}")
        print(f"      Texto: {res.document.text[:100]}...")

## Part II: Filtering by metadata

In [ ]:
from datasets import load_dataset

class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts)
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))

    def search(self, 
               query: str, 
               top_k: int = 5, 
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        
        if self.embeddings is None or len(self.documents) == 0:
            return []

        # PRE-FILTRADO: Obtener índices que cumplen el criterio
        valid_indices = []
        if metadata_filter:
            for idx, doc in enumerate(self.documents):
                match = True
                for key, val in metadata_filter.items():
                    if doc.metadata.get(key) != val:
                        match = False
                        break
                if match:
                    valid_indices.append(idx)
                    
            if not valid_indices:
                return [] # Ningún documento cumple el filtro
            
            target_embeddings = self.embeddings[valid_indices]
            target_docs = [self.documents[i] for i in valid_indices]
        else:
            target_embeddings = self.embeddings
            target_docs = self.documents

        # Operaciones de similitud solo sobre los embeddings filtrados
        query_embedding = self.embedding_model.encode([query])[0]
        
        dot_products = np.dot(target_embeddings, query_embedding)
        norms_docs = norm(target_embeddings, axis=1)
        norm_query = norm(query_embedding)
        norms_docs[norms_docs == 0] = 1e-10
        
        similarities = dot_products / (norms_docs * norm_query)
        
        # Obtener top_k de la sub-lista
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        return [SearchResult(float(similarities[idx]), target_docs[idx]) for idx in top_indices]

# ==========================================
# Ejecución y Pruebas - Parte II
# ==========================================

print("\nInicializando FilteredVectorStore...")
filtered_store = FilteredVectorStore(model)

print("Cargando AG News dataset (muestra de 1000 documentos para velocidad)...")
# Mapeo de etiquetas de AG News
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
dataset = load_dataset("ag_news", split="train[:1000]")

news_docs = []
for item in dataset:
    metadata = {
        "category": label_map[item["label"]],
    }
    # El texto de AG News
    doc = Document(text=item["text"], metadata=metadata)
    news_docs.append(doc)

print(f"Indexando {len(news_docs)} noticias con metadatos...")
filtered_store.add_documents(news_docs)

consultas_noticias = [
    {"q": "New discoveries in space exploration", "filter": {"category": "Sci/Tech"}},
    {"q": "Stock market drops significantly", "filter": {"category": "Business"}},
    {"q": "Championship final match results", "filter": {"category": "Sports"}},
    {"q": "Software update causes massive outage", "filter": {"category": "Sci/Tech"}},
    {"q": "Elections happening in Europe", "filter": {"category": "World"}}
]

print("\n--- Resultados Parte II (Con Filtros) ---")
for item in consultas_noticias:
    q = item["q"]
    f = item["filter"]
    print(f"\nConsulta: '{q}' | Filtro: {f}")
    resultados = filtered_store.search(q, top_k=2, metadata_filter=f)
    
    if not resultados:
        print("  Sin resultados que coincidan con el filtro.")
    
    for i, res in enumerate(resultados):
        cat = res.document.metadata.get('category', 'Unknown')
        print(f"  [{i+1}] Score: {res.score:.4f} | Categoría: {cat}")
        print(f"      Texto: {res.document.text[:100]}...")